In [ ]:
import os, pandas as pd
os.makedirs("/content/sph", exist_ok=True)
!wget -q -O /content/sph/metadata.csv https://ndownloader.figshare.com/files/34793152
!wget -q -O /content/sph/code.csv     https://ndownloader.figshare.com/files/32630954

meta = pd.read_csv("/content/sph/metadata.csv")
code = pd.read_csv("/content/sph/code.csv").astype(str)
print(meta.columns.tolist(), meta.shape)
print(code[code["Description"].str.contains("fibrillation|flutter", case=False)])

['ECG_ID', 'AHA_Code', 'Patient_ID', 'Age', 'Sex', 'N', 'Date'] (25770, 7)
  Category Code          Description
8        E   50  Atrial fibrillation
9        E   51       Atrial flutter


In [ ]:
AF_CODE = "50"

def codes(s):
    return {c.split("+")[0].strip() for c in str(s).split(";")}

meta["has_af"] = meta["AHA_Code"].apply(lambda s: AF_CODE in codes(s))
print("AF records:", meta.has_af.sum(), f"({meta.has_af.mean():.1%})")
print("AF patients:", meta.loc[meta.has_af, "Patient_ID"].nunique())

AF records: 675 (2.6%)
AF patients: 634


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

False no GPU


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Inspect Folder

In [ ]:
SPH_DIR = "/content/drive/MyDrive/SPH_DATA"

In [ ]:
import os, glob
SPH_DIR = "/content/drive/MyDrive/SPH_DATA"     # change if it lives elsewhere
print(os.listdir(SPH_DIR)[:20])
#print("h5 files:", len(glob.glob(f"{SPH_DIR}/**/*.h5", recursive=True)))
#print("archives:", glob.glob(f"{SPH_DIR}/**/*.tar*", recursive=True) + glob.glob(f"{SPH_DIR}/**/*.zip", recursive=True))
#print("csvs:", glob.glob(f"{SPH_DIR}/**/*.csv", recursive=True))

['metadata.csv', 'code.csv', 'records', 'Independent_AFib_Statistical_Report.xlsx']


In [ ]:
!git clone https://github.com/Jwoo5/fairseq-signals.git
%cd /content/fairseq-signals
!pip install --editable ./
!pip install huggingface_hub
%cd /content

Cloning into 'fairseq-signals'...
remote: Enumerating objects: 4447, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 4447 (delta 203), reused 197 (delta 197), pack-reused 4187 (from 2)
Receiving objects: 100% (4447/4447), 5.41 MiB | 13.78 MiB/s, done.
Resolving deltas: 100% (2916/2916), done.
/content/fairseq-signals
Obtaining file:///content/fairseq-signals
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.1 MB/s eta 0:00:0

In [ ]:
!git clone https://github.com/Jwoo5/fairseq-signals.git
%cd /content/fairseq-signals
!pip install --editable ./

fatal: destination path 'fairseq-signals' already exists and is not an empty directory.
/content/fairseq-signals
Obtaining file:///content/fairseq-signals
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fairseq_signals (pyproject.toml) ... done
  Created wheel for fairseq_signals: filename=fairseq_signals-1.0.0a0+f8f0ff1-0.editable-cp313-cp313-linux_x86_64.whl size=6634 sha256=4cdf770dfe49f75ffb8ebd3ae05e0b1c19e14c72b158d9abca40edc7576f2c24
  Stored in directory: /tmp/pip-ephem-wheel-cache-bx4z53cz/wheels/f4/82/d5/6a25fc7d41110e9d01841054e2acd53c9ccb2cf18dd87be938
Successfully built fairseq_signals
  Attempting uninstall: fairseq_signals
    Found existing installation: fairseq_signals 1.0.0a0+f8f0ff1
    Uninstalling fairseq_signals-1.0.0a0+f8f0ff1:
      Succes

# Gets ECG-FM weights

In [ ]:
from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(repo_id="wanglab/ecg-fm", filename="mimic_iv_ecg_physionet_pretrained.pt")
print(ckpt_path)

mimic_iv_ecg_physionet_pretrained.pt: reconstructing file:   0%|          |  0.00B / 1.09GB            

mimic_iv_ecg_physionet_pretrained.pt: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/models--wanglab--ecg-fm/snapshots/584219ea492cdeef2e19ffbdf9c6ecc874ba427e/mimic_iv_ecg_physionet_pretrained.pt


# load metadata and match to files

In [ ]:
import pandas as pd, numpy as np, h5py

# Where the .h5 files are: Drive, or the local extraction from Step 3
H5_ROOT = SPH_DIR                 # use "/content/sph_local" if you extracted the tar

found = glob.glob(f"{SPH_DIR}/**/metadata.csv", recursive=True)
if found:
    meta_path = found[0]
else:
    os.makedirs("/content/sph_meta", exist_ok=True)
    meta_path = "/content/sph_meta/metadata.csv"
    !wget -q -O /content/sph_meta/metadata.csv https://ndownloader.figshare.com/files/34793152

meta = pd.read_csv(meta_path)
print(meta.columns.tolist(), meta.shape)

paths = {os.path.splitext(os.path.basename(p))[0]: p
         for p in glob.glob(f"{H5_ROOT}/**/*.h5", recursive=True)}
meta["rec"]  = meta["ECG_ID"].astype(str).str.replace(".h5", "", regex=False)
meta["path"] = meta["rec"].map(paths)
print("metadata rows:", len(meta), "| rows with a file:", meta["path"].notna().sum())
meta = meta[meta["path"].notna()].reset_index(drop=True)

['ECG_ID', 'AHA_Code', 'Patient_ID', 'Age', 'Sex', 'N', 'Date'] (25770, 7)
metadata rows: 25770 | rows with a file: 25770


# Check one record and define preprocessing

In [ ]:
with h5py.File(meta.loc[0, "path"], "r") as f:
    print(list(f.keys()))
    x = f["ecg"][:]
print(x.shape, x.dtype)       # expect (12, L), L = 500 * seconds

['ecg']
(12, 5000) float16


In [ ]:
WIN = 2500   # 5 s at 500 Hz

def load_windows(path):
    """-> (tensor (n,12,2500), flat_lead_flag) or None"""
    with h5py.File(path, "r") as f:
        x = f["ecg"][:].astype(np.float32)
    if x.shape[0] != 12 and x.shape[1] == 12:      # handle (L,12) just in case
        x = x.T
    x = np.nan_to_num(x)
    sd = x.std(axis=1, keepdims=True)
    flat = bool((sd < 1e-6).any())                 # any flat lead -> flag for QC
    x = (x - x.mean(axis=1, keepdims=True)) / (sd + 1e-8)
    n = x.shape[1] // WIN
    if n == 0:
        return None
    segs = np.stack([x[:, i*WIN:(i+1)*WIN] for i in range(n)])
    return torch.from_numpy(segs), flat

segs, flat = load_windows(meta.loc[0, "path"])
print(segs.shape, flat)       # e.g. torch.Size([2, 12, 2500]) False

torch.Size([2, 12, 2500]) False


# Load Model

In [ ]:
from fairseq_signals.models import build_model_from_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model_from_checkpoint(checkpoint_path=ckpt_path)
model = model.to(device).eval()
print(device)

cpu


# Embeddings Functions

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def embed_quiet(model, x):
    x = x.to(device)
    with torch.no_grad():
        out = model.extract_features(source=x, padding_mask=None, mask=False)
    h = out["x"] if isinstance(out, dict) else out[0]
    if h.shape[0] != x.shape[0] and h.shape[1] == x.shape[0]:
        h = h.transpose(0, 1)
    return h.mean(dim=1).cpu().numpy()

def _load(item):
    rec, path = item
    try:
        r = load_windows(path)
        return None if r is None else (rec, r[0], r[1])
    except Exception as e:
        print("skip", rec, e)
        return None

def embed_records(df_sub, batch_records=32, workers=16, gpu_bs=256):
    ids, embs, nwin, flat = [], [], [], []
    items = list(zip(df_sub["rec"], df_sub["path"]))
    with ThreadPoolExecutor(workers) as ex:
        for i in tqdm(range(0, len(items), batch_records)):
            res = [r for r in ex.map(_load, items[i:i+batch_records]) if r is not None]
            if not res:
                continue
            counts = [r[1].shape[0] for r in res]
            xb = torch.cat([r[1] for r in res])
            e = np.concatenate([embed_quiet(model, xb[j:j+gpu_bs])
                                for j in range(0, len(xb), gpu_bs)])
            offs = np.cumsum([0] + counts)
            for k, r in enumerate(res):
                embs.append(e[offs[k]:offs[k+1]].mean(axis=0))   # average windows -> one vector per record
                ids.append(r[0]); nwin.append(counts[k]); flat.append(r[2])
    return np.array(ids), np.stack(embs), np.array(nwin), np.array(flat)

## First 64 as Test

In [ ]:
t_ids, t_E, t_nw, t_flat = embed_records(meta.iloc[:64])
print(t_E.shape)                 # expect (64, 768)
print(t_ids[:5])
print(np.isnan(t_E).any())       # expect False
print("windows per record:", t_nw.min(), "-", t_nw.max(), "| flat-lead records:", t_flat.sum())

  0%|          | 0/2 [00:00<?, ?it/s]

(64, 768)
['A00001' 'A00002' 'A00003' 'A00004' 'A00005']
False
windows per record: 2 - 4 | flat-lead records: 0


## Full Run

In [ ]:
CHUNK = 2000
OUT_DIR = "/content/drive/MyDrive/sph_ecgfm"
os.makedirs(OUT_DIR, exist_ok=True)

for start in range(0, len(meta), CHUNK):
    end = min(start + CHUNK, len(meta))
    f_out = f"{OUT_DIR}/chunk_{start}_{end}.npz"
    if os.path.exists(f_out):
        print("skip (already done):", start, end)
        continue
    ids, E, nw, fl = embed_records(meta.iloc[start:end])
    np.savez(f_out, ids=ids, emb=E, nwin=nw, flat=fl)
    print("saved", start, end, E.shape)

skip (already done): 0 2000
skip (already done): 2000 4000
skip (already done): 4000 6000
skip (already done): 6000 8000
skip (already done): 8000 10000
skip (already done): 10000 12000
skip (already done): 12000 14000
skip (already done): 14000 16000
skip (already done): 16000 18000
skip (already done): 18000 20000
skip (already done): 20000 22000
skip (already done): 22000 24000


  0%|          | 0/56 [00:00<?, ?it/s]

saved 24000 25770 (1770, 768)


## Merge and sanity check

In [ ]:
import re
files = sorted(glob.glob(f"{OUT_DIR}/chunk_*.npz"),
               key=lambda p: int(re.search(r"chunk_(\d+)_", p).group(1)))
parts = [np.load(f, allow_pickle=True) for f in files]
ids_all  = np.concatenate([p["ids"]  for p in parts])
E_all    = np.concatenate([p["emb"]  for p in parts])
nwin_all = np.concatenate([p["nwin"] for p in parts])
flat_all = np.concatenate([p["flat"] for p in parts])

print(E_all.shape)                          # expect (~25770, 768)
print(len(ids_all), len(set(ids_all)))      # both the same = no duplicates
print(np.isnan(E_all).any())                # expect False
print("windows/record: mean", nwin_all.mean().round(2), "max", nwin_all.max(),
      "| records with a flat lead:", flat_all.sum())

np.savez(f"{OUT_DIR}/sph_ecgfm_all.npz", ids=ids_all, emb=E_all, nwin=nwin_all, flat=flat_all)

(25770, 768)
25770 25770
False
windows/record: mean 2.08 max 11 | records with a flat lead: 0


## Label Check - should see 675 AF records

In [ ]:
code_found = glob.glob(f"{SPH_DIR}/**/code.csv", recursive=True)
if code_found:
    code = pd.read_csv(code_found[0]).astype(str)
else:
    !wget -q -O /content/sph_meta/code.csv https://ndownloader.figshare.com/files/32630954
    code = pd.read_csv("/content/sph_meta/code.csv").astype(str)
print(code[code["Description"].str.contains("fibrillation|flutter", case=False)])

AF_CODE, FLUTTER_CODE = "50", "51"     # <-- set from the printout (AF_CODE = the one that gave 675)

def codes(s): return {c.split("+")[0].strip() for c in str(s).split(";")}
M = meta.set_index("rec").loc[ids_all]
has_af   = M["AHA_Code"].apply(lambda s: AF_CODE in codes(s)).values
has_flut = M["AHA_Code"].apply(lambda s: FLUTTER_CODE in codes(s)).values
print("AF records:", has_af.sum(), "| AF patients:", M.loc[has_af, "Patient_ID"].nunique())
print("flutter-only (to exclude from negatives):", (has_flut & ~has_af).sum())

  Category Code          Description
8        E   50  Atrial fibrillation
9        E   51       Atrial flutter
AF records: 675 | AF patients: 634
flutter-only (to exclude from negatives): 99
